# inbalanced datass

In [2]:
# imports
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import os
os.chdir('/home/jerry/Documents/master/Master/Machine learning - PART IV/Data')
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import *

In [3]:
df = pd.read_csv('diabetes.csv')
df.drop_duplicates(inplace=True)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,Pedigree,Age,Diabetes
0,10,129,76,28,122,35.9,0.280,39,0
1,4,84,90,23,56,39.5,0.159,25,0
2,0,84,82,31,125,38.2,0.233,23,0
3,9,134,74,33,60,25.9,0.460,81,0
4,0,124,56,13,105,21.8,0.452,21,0


In [4]:
df.groupby('Diabetes').size() # vemos que el desbalanceo es considerable. 500 de clase 0y 21 de clase 1
# por lo tanto lso algoritmos clasicos no funcionan



Diabetes
0    500
1     21
dtype: int64

In [5]:
# enfoque clasico
seed = 99
models = []
models.append(('lr', LogisticRegression(random_state=seed)))
models.append(('knn', KNeighborsClassifier()))
models.append(('rf', RandomForestClassifier(random_state=seed)))

In [6]:
x = df.drop(columns='Diabetes')
y = df['Diabetes']


x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=.2, random_state=seed,stratify=y) # esto es para mantener siempre la misma proporcion de y (stratify)
print('y_test')
print(' numero de 0', np.count_nonzero(y_test == 0))
print(' numero de 1', np.count_nonzero(y_test == 1))

# como vemos hay muy pocos unos entonces para crear con el algoritmo clasicos bien, de todos modos probamos


y_test
 numero de 0 101
 numero de 1 4


In [7]:
# probamos entrenando modelos calsicos
for name , model in models:
    model.fit(x_train, y_train)
    y_hat = model.predict(x_test)

    acc = accuracy_score(y_test,y_hat)
    rec = recall_score(y_test,y_hat)
    pre = precision_score(y_test,y_hat)
    print(f'model: {model} acc : {acc} pre : {pre} rec : {rec}')

# enteoria nos salen modelos super buenos. pero no pdoemso ir a pro con un modelo de estos teniendo un dataset desbalanceado.
# vemos la amtriz de confusion

confusion_matrix(y_true=y_test,y_pred=y_hat) # nos lo esta clasificacion todos a 0 menos 1 a 1, entonces no tiene mucho sentido funcioann muy muy mal. ## si vemos tmb el resto de las metricas las vemos muy mal

model: LogisticRegression(random_state=99) acc : 0.9428571428571428 pre : 0.0 rec : 0.0
model: KNeighborsClassifier() acc : 0.9523809523809523 pre : 0.0 rec : 0.0
model: RandomForestClassifier(random_state=99) acc : 0.9523809523809523 pre : 0.0 rec : 0.0


array([[100,   1],
       [  4,   0]])

## estrategia 1: stratificaction

In [8]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5,random_state=seed, shuffle=True)


for name, model in models:

    grid_model = GridSearchCV(estimator=model, cv=cv, param_grid={})


    grid_model.fit(x_train, y_train)
    y_hat = grid_model.predict(x_test)

    acc = accuracy_score(y_test,y_hat)
    rec = recall_score(y_test,y_hat)
    pre = precision_score(y_test,y_hat)
    print(f'model: {model} acc : {acc} pre : {pre} rec : {rec}') # tmpc emos conseguido nada nuevo. no hemso evolucionado gran cosa. podria haber funcionado pero con datasets tan desbalanceados, no funciona muy bien. pero con 70/30 igual funcionaria o con 60/40

model: LogisticRegression(random_state=99) acc : 0.9428571428571428 pre : 0.0 rec : 0.0
model: KNeighborsClassifier() acc : 0.9523809523809523 pre : 0.0 rec : 0.0
model: RandomForestClassifier(random_state=99) acc : 0.9523809523809523 pre : 0.0 rec : 0.0


## oversampling

In [9]:
from imblearn.over_sampling import RandomOverSampler

In [ ]:
ros = RandomOverSampler(random_state=seed)
x_resamplec, y_resampled = ros.fit_resample(x_train, y_train) # sale como un train test split igual, pero salen balanceados. vamos a verlo

print('y train')
print('nuemro de 0s',np.count_nonzero(y_train==0))
print('nuemro de 1s',np.count_nonzero(y_train==1))

print('y reesampled')
print('nuemro de 0s',np.count_nonzero(y_resampled==0))
print('nuemro de 1s',np.count_nonzero(y_resampled==1)) ## como podemso ver nos ha añadido muchos mas registros de 1 para que peudan llegar a ser iguales las cantidades y proporciines basicamente duplicando registros. el modelo aprende de manera sesgada pero es mejor q nada y es una solcuion

y train
nuemro de 0s 399
nuemro de 1s 17
y reesampled
nuemro de 0s 399
nuemro de 1s 399


In [12]:
for name, model in models:

    grid_model = GridSearchCV(estimator=model, cv=cv, param_grid={})


    grid_model.fit(x_resamplec, y_resampled)
    y_hat = grid_model.predict(x_test)

    acc = accuracy_score(y_test,y_hat)
    rec = recall_score(y_test,y_hat)
    pre = precision_score(y_test,y_hat)
    print(f'model: {model} acc : {acc} pre : {pre} rec : {rec}') # tmpc emos conseguido nada nuevo. no hemso evolucionado gran cosa. podria haber funcionado pero con datasets tan desbalanceados, no funciona muy bien. pero con 70/30 igual funcionaria o con 60/40

model: LogisticRegression(random_state=99) acc : 0.7904761904761904 pre : 0.05 rec : 0.25
model: KNeighborsClassifier() acc : 0.8857142857142857 pre : 0.1 rec : 0.25
model: RandomForestClassifier(random_state=99) acc : 0.9619047619047619 pre : 0.0 rec : 0.0


## estrategia 3: random undersamplng

In [13]:
from imblearn.under_sampling import RandomUnderSampler

In [ ]:
rus = RandomUnderSampler(random_state=seed)
x_resampler, y_resampler = rus.fit_resample(x_train, y_train)

print('y train')
print('nuemro de 0s',np.count_nonzero(y_train==0))
print('nuemro de 1s',np.count_nonzero(y_train==1))

print('y reesampled')
print('nuemro de 0s',np.count_nonzero(y_resampler==0))
print('nuemro de 1s',np.count_nonzero(y_resampler==1))
# en este caso, lo que hacemos es reducir los datos en lugar de aumentarlos para q la proprocion se ala misma q la menor

y train
nuemro de 0s 399
nuemro de 1s 17
y reesampled
nuemro de 0s 17
nuemro de 1s 17


In [ ]:
for name, model in models:

    grid_model = GridSearchCV(estimator=model, cv=cv, param_grid={})


    grid_model.fit(x_resampler, y_resampler)
    y_hat = grid_model.predict(x_test)

    acc = accuracy_score(y_test,y_hat)
    rec = recall_score(y_test,y_hat)
    pre = precision_score(y_test,y_hat)
    print(f'model: {model} acc : {acc} pre : {pre} rec : {rec}') 

## de hecho ahora el rf se ha movido algo. no son perfectos pero algo esta haciendo y moviendo. estos algoritmos clasicos estan 'despertando'

model: LogisticRegression(random_state=99) acc : 0.6476190476190476 pre : 0.07692307692307693 rec : 0.75
model: KNeighborsClassifier() acc : 0.6476190476190476 pre : 0.05405405405405406 rec : 0.5
model: RandomForestClassifier(random_state=99) acc : 0.7333333333333333 pre : 0.1 rec : 0.75


## smote

In [17]:
from imblearn.over_sampling import SMOTE


In [ ]:
smote = SMOTE(random_state=seed)
x_resampled, y_resampled = smote.fit_resample(x_train, y_train)

print('y train')
print('nuemro de 0s',np.count_nonzero(y_train==0))
print('nuemro de 1s',np.count_nonzero(y_train==1))

print('y reesampled')
print('nuemro de 0s',np.count_nonzero(y_resampled==0))
print('nuemro de 1s',np.count_nonzero(y_resampled==1)) ## han salido tantos 1 como 0


y train
nuemro de 0s 399
nuemro de 1s 17
y reesampled
nuemro de 0s 399
nuemro de 1s 399


In [ ]:
for name, model in models:

    grid_model = GridSearchCV(estimator=model, cv=cv, param_grid={})


    grid_model.fit(x_resampler, y_resampler)
    y_hat = grid_model.predict(x_test)

    acc = accuracy_score(y_test,y_hat)
    rec = recall_score(y_test,y_hat)
    pre = precision_score(y_test,y_hat)
    print(f'model: {model} acc : {acc} pre : {pre} rec : {rec}') 

## con esto empieza a funcionar bastante bien los algoritmos el rf parece q ya va bastante bien.

#vamosa ver como queda la cm:

confusion_matrix(y_true=y_test,y_pred=y_hat)  # aprece que ya va bastante mejor, pero aun asi seguramente s epueda mejorar sin usar estos algoritmos clasicos


model: LogisticRegression(random_state=99) acc : 0.6476190476190476 pre : 0.07692307692307693 rec : 0.75
model: KNeighborsClassifier() acc : 0.6476190476190476 pre : 0.05405405405405406 rec : 0.5
model: RandomForestClassifier(random_state=99) acc : 0.7333333333333333 pre : 0.1 rec : 0.75


array([[74, 27],
       [ 1,  3]])

## estrategia 5, algoritmos para datasets desbalanceados

In [21]:
from imblearn.ensemble import BalancedRandomForestClassifier

In [ ]:
model = BalancedRandomForestClassifier(
    sampling_strategy='all',
    random_state=seed,
    replacement=True,
    max_depth=10,
    bootstrap=True
)

model.fit(x_train,y_train)
y_hat = model.predict(x_test)
acc = accuracy_score(y_test,y_hat)
rec = recall_score(y_test,y_hat)
pre = precision_score(y_test,y_hat)
print(f'model: {model} acc : {acc} pre : {pre} rec : {rec}') 

## con un dataser mas grande este modelo podria haber funcionado bastante mejor. parece que va decentemente bien. vamos a ver la cm
confusion_matrix(y_true=y_test,y_pred=y_hat)  
# aprece que lo mejro que ha funcionado por ahora es el smote pero este dataset es mas chico, normalmentel se debe probar todo para probar cual va mejor 

model: BalancedRandomForestClassifier(bootstrap=True, max_depth=10, random_state=99) acc : 0.7523809523809524 pre : 0.10714285714285714 rec : 0.75


array([[76, 25],
       [ 1,  3]])

## estrategia 6: class weitght

In [ ]:
model = RandomForestClassifier(
    n_estimators=3,
    max_depth=5,
    random_state=seed,
    class_weight={0:0.5,1:12}
)

model.fit(x_train,y_train)
y_hat = model.predict(x_test)
acc = accuracy_score(y_test,y_hat)
rec = recall_score(y_test,y_hat)
pre = precision_score(y_test,y_hat)
print(f'model: {model} acc : {acc} pre : {pre} rec : {rec}') 
# aqui estamos viendo q puede q tengamos resultados muy buenos con este modelo con el balanceo de peso.
## pero lo buenod e estoe s que lo podemos poner con gridsearch y poner el balanceo que nosotros queremos. por ejemplo poniendo class_weigth={0:0.5,1:12}


model: RandomForestClassifier(class_weight='balanced', max_depth=5, n_estimators=3,
                       random_state=99) acc : 0.8 pre : 0.13043478260869565 rec : 0.75


In [ ]:
len(x_train) / (2*np.bincount(y_train)) ## esto con balanced, a la clase minoritaria el da peso 0.5 y a la mayoritaria le da peso 12 y con esto entrena el modelo

array([ 0.52130326, 12.23529412])